# Phase 18D Notebook 02: Verify Kaggle Readiness\n\nThis notebook verifies OASIS, cohort mappings, provenance, privacy, folds, target partitions, concept/anatomy reuse, and readiness-state transitions. It performs no training, predictive evaluation, publication analysis, or Phase 19 work.

In [ ]:
from __future__ import annotations
import hashlib
import json
import os
import random
from pathlib import Path

PADA3DACB_SUBJECT_HMAC_KEY = os.environ.get("PADA3DACB_SUBJECT_HMAC_KEY")
real_execution_authorized = False
publication_authorized = False
phase_19_forbidden = True
INPUT_ROOT = Path("/kaggle/input")
BINDING = Path("/kaggle/working/pada3dacb_binding")
MODEL_READY = Path("/kaggle/working/pada3dacb_model_ready")
BUNDLE = Path("/kaggle/working/pada3dacb_readiness_bundle")
if not BINDING.is_dir() or not MODEL_READY.is_dir():
    raise RuntimeError("READINESS_INPUT_MISSING: execute Notebook 00 and 01 first")

def fail(reason: str) -> None:
    raise RuntimeError(reason)

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

oasis_path = os.environ.get("PADA3DACB_OASIS_EVIDENCE_JSON")
if not oasis_path or not Path(oasis_path).is_file():
    fail("OASIS_EVIDENCE_MISSING")
oasis = json.loads(Path(oasis_path).read_text(encoding="utf-8"))
expected = {"visits": 436, "canonical_persons": 416, "repeated_visits_excluded": 20, "CN": 316, "Impaired": 100}
expected_mapping = {"0": "CN", "0.5": "Impaired", "1": "Impaired", "2": "Impaired"}
if oasis.get("counts") != expected or oasis.get("mapping") != expected_mapping:
    fail("BLOCKED_COHORT_MISMATCH")
adni = json.loads((MODEL_READY / "adni_binary_manifest.json").read_text(encoding="utf-8"))
oasis_manifest_path = Path(os.environ.get("PADA3DACB_OASIS_BINARY_MANIFEST", ""))
if not oasis_manifest_path.is_file():
    fail("OASIS_BINARY_MANIFEST_MISSING")
oasis_manifest = json.loads(oasis_manifest_path.read_text(encoding="utf-8"))
for cohort, manifest in (("ADNI", adni), ("OASIS", oasis_manifest)):
    if manifest.get("task_id") != "cn_vs_impaired" or manifest.get("class_order") != ["CN", "Impaired"] or not manifest.get("persons"):
        fail(f"{cohort}_BINARY_MANIFEST_INVALID")
artifact_root = os.environ.get("PADA3DACB_CONCEPT_ANATOMY_ROOT")
if not artifact_root or not Path(artifact_root).is_dir():
    fail("APPROVED_CONCEPT_ANATOMY_ROOT_MISSING")
artifact_records = {}
for logical in ("c_target", "g_bar", "normalizer", "roi_order", "atlas", "masks", "jacobian"):
    path = Path(artifact_root) / logical
    if not path.is_file():
        fail(f"APPROVED_ARTIFACT_MISSING:{logical}")
    artifact_records[logical] = {"path_name": logical, "sha256": sha256(path), "refit": False, "regenerated": False}

def split(persons):
    persons = sorted(persons, key=lambda item: item["subject_hash"])
    shuffled = list(persons)
    random.Random(42).shuffle(shuffled)
    folds = {f"fold_{index}": [] for index in range(5)}
    for label in (0, 1):
        tokens = [item["subject_hash"] for item in persons if item["binary_label"] == label]
        for index, token in enumerate(tokens):
            folds[f"fold_{index % 5}"].append(token)
    cut = max(1, min(len(shuffled) - 1, round(len(shuffled) * 0.8)))
    adaptation = [item["subject_hash"] for item in shuffled[:cut]]
    evaluation = [item["subject_hash"] for item in shuffled[cut:]]
    if set(adaptation) & set(evaluation):
        fail("TARGET_PARTITION_OVERLAP")
    return folds, adaptation, evaluation

persons = adni["persons"] + oasis_manifest["persons"]
folds, adaptation, evaluation = split(persons)
BUNDLE.mkdir(parents=True, exist_ok=True)
(BUNDLE / "cohort_manifest.json").write_text(json.dumps({"task_id": "cn_vs_impaired", "class_order": ["CN", "Impaired"], "persons": persons}, indent=2), encoding="utf-8")
(BUNDLE / "splits_manifest.json").write_text(json.dumps({"source_folds": folds, "target_adaptation": adaptation, "target_evaluation": evaluation, "target_firewall": {"status": "pass", "fields": ["subject_hash", "cohort"]}, "seed": 42}, indent=2), encoding="utf-8")
(BUNDLE / "oasis_verification.json").write_text(json.dumps({"counts": expected, "mapping": expected_mapping}, indent=2), encoding="utf-8")
(BUNDLE / "concept_anatomy_reuse.json").write_text(json.dumps({"refit": False, "regenerated": False, "artifacts": artifact_records}, indent=2), encoding="utf-8")
try:
    import torch
    synthetic = torch.randn(4, 8, requires_grad=True)
    synthetic.square().mean().backward()
    if synthetic.grad is None or not bool(torch.isfinite(synthetic.grad).all()):
        fail("SYNTHETIC_PROBE_FAILED")
except ImportError:
    fail("PYTORCH_MISSING")
(BUNDLE / "readiness_state.json").write_text(json.dumps({"state": "KAGGLE_READINESS_EVIDENCE_PRODUCED", "authorization_flags": {"authorized": False, "real_execution_authorized": False, "freeze_approved": False, "publication_authorized": False, "phase_19_forbidden": True}}, indent=2), encoding="utf-8")
print("KAGGLE_READINESS_EVIDENCE_PRODUCED")
